# DCGAN on SVHN — Improved Version

This notebook is an improved version of the DCGAN SVHN implementation, addressing **discriminator dominance** observed in the original training run.

### Improvements applied (all within the original TF tutorial structure):
| Fix | Why |
|---|---|
| One-sided label smoothing (1.0 → 0.9) | Stops discriminator from being overconfident early |
| Generator updated 2× per discriminator step | Rebalances adversarial dynamic |
| Annealing input noise on discriminator | Slows early discriminator dominance |
| Adam `beta_1=0.5` for both optimizers | Standard GAN recommendation; reduces momentum overshoot |
| Deeper generator (added 64-filter layer) | More capacity for color/detail in SVHN images |

## 1. Setup & Imports

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')

In [ ]:
import glob
import imageio
import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
from tensorflow.keras import layers
import time
from IPython import display

try:
    import tensorflow_datasets as tfds
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'tensorflow-datasets', '-q'])
    import tensorflow_datasets as tfds

print('All imports successful.')

## 2. Load and Preprocess SVHN

Same as original — SVHN via `tensorflow_datasets`, normalized to `[-1, 1]`.

In [ ]:
ds_train, ds_info = tfds.load(
    'svhn_cropped',
    split='train',
    as_supervised=False,
    with_info=True
)

print(f'Training examples : {ds_info.splits["train"].num_examples}')
print(f'Image shape       : {ds_info.features["image"].shape}')

In [ ]:
IMAGE_SHAPE = (32, 32, 3)
BUFFER_SIZE = 73257
BATCH_SIZE  = 256

def preprocess(sample):
    image = tf.cast(sample['image'], tf.float32)
    image = (image - 127.5) / 127.5  # [-1, 1]
    return image

train_dataset = (
    ds_train
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)   # drop_remainder keeps batch size constant
    .prefetch(tf.data.AUTOTUNE)
)

print('Dataset pipeline ready.')

In [ ]:
# Visualize sample training images
sample_batch = next(iter(train_dataset))

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(np.clip((sample_batch[i].numpy() + 1.0) / 2.0, 0, 1))
    ax.axis('off')
plt.suptitle('SVHN Training Samples', fontsize=14)
plt.tight_layout()
plt.savefig('svhn_samples.png', dpi=100)
plt.show()

## 3. Generator — Improved

**Change vs. original:** Added an extra `Conv2DTranspose` layer with 64 filters between the 128-filter layer and the RGB output. This gives the generator more capacity to learn fine details and color gradients before committing to the final 3-channel output.

Upsampling path: `4×4 → 8×8 → 16×16 → 32×32 → 32×32` (last step same spatial size, refines features)

In [ ]:
def make_generator_model():
    model = tf.keras.Sequential(name='Generator')

    # Project noise to 4×4 spatial map
    model.add(layers.Dense(4 * 4 * 512, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    model.add(layers.Reshape((4, 4, 512)))

    # 4×4 → 8×8
    model.add(layers.Conv2DTranspose(256, (5, 5), strides=(2, 2),
                                     padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # 8×8 → 16×16
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2),
                                     padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # 16×16 → 32×32  (added layer — refines features at full resolution)
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2),
                                     padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # 32×32 — final RGB output (stride=1: same spatial size, just maps to 3 channels)
    model.add(layers.Conv2DTranspose(3, (5, 5), strides=(1, 1),
                                     padding='same', use_bias=False,
                                     activation='tanh'))
    assert model.output_shape == (None, 32, 32, 3)

    return model

generator = make_generator_model()
generator.summary()

In [ ]:
# Sanity-check: untrained output
noise = tf.random.normal([1, 100])
gen_img = generator(noise, training=False)
print(f'Generator output shape: {gen_img.shape}')  # (1, 32, 32, 3)

plt.figure(figsize=(3, 3))
plt.imshow(np.clip((gen_img[0].numpy() + 1.0) / 2.0, 0, 1))
plt.title('Untrained generator (noise)')
plt.axis('off')
plt.show()

## 4. Discriminator — Same as original

The discriminator architecture is unchanged from the v1 notebook. The training balance fixes applied later (label smoothing, input noise, 2× generator updates) are sufficient to stop it from dominating without changing its capacity.

In [ ]:
def make_discriminator_model():
    model = tf.keras.Sequential(name='Discriminator')

    # 32×32 → 16×16
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2),
                            padding='same', input_shape=IMAGE_SHAPE))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # 16×16 → 8×8
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # 8×8 → 4×4
    model.add(layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))  # Real/fake logit

    return model

discriminator = make_discriminator_model()
discriminator.summary()

## 5. Loss Functions — Improved

**Change 1 — One-sided label smoothing:**
Real labels changed from `1.0` → `0.9`. This prevents the discriminator from becoming overconfident (saturating its sigmoid), which was causing near-zero loss in the first few epochs of the original run. Fake labels remain `0.0` (only one side is smoothed).

**Change 2 — Adam `beta_1=0.5` (both optimizers):**
The default `beta_1=0.9` carries too much momentum in adversarial training, allowing the discriminator to overshoot and lock in a dominant position before the generator can adapt. `beta_1=0.5` is the standard recommendation from the DCGAN paper.

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# ── IMPROVED: one-sided label smoothing ──────────────────────────────────────
LABEL_SMOOTH = 0.9   # real labels: 1.0 → 0.9  |  fake labels stay 0.0

def discriminator_loss(real_output, fake_output):
    # Smooth real labels to 0.9 to prevent discriminator overconfidence
    real_loss = cross_entropy(
        tf.ones_like(real_output) * LABEL_SMOOTH, real_output)
    fake_loss = cross_entropy(
        tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    # Generator wants discriminator to label fakes as real
    return cross_entropy(tf.ones_like(fake_output), fake_output)

# ── IMPROVED: beta_1=0.5 (standard GAN recommendation) ──────────────────────
generator_optimizer     = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)

print(f'Label smoothing   : real labels → {LABEL_SMOOTH}')
print(f'Optimizer beta_1  : 0.5 (was 0.9)')

## 6. Checkpoints

In [ ]:
checkpoint_dir    = './training_checkpoints_v2'
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')
checkpoint = tf.train.Checkpoint(
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator=generator,
    discriminator=discriminator
)
print(f'Checkpoint dir: {checkpoint_dir}')

## 7. Training Loop — Improved

Three targeted changes to `train_step()`:

**Change 1 — Annealing input noise on the discriminator:**
Gaussian noise is added to both real and fake images fed into the discriminator. The standard deviation starts at `0.1` and linearly decays to `0.0` by epoch 30. This prevents the discriminator from memorizing exact pixel patterns in the early epochs, keeping its loss from collapsing to near-zero.

**Change 2 — Generator updated twice per step:**
The generator gradient update is applied twice inside each `train_step` call. This directly compensates for the discriminator's early advantage, giving the generator twice as many gradient steps to catch up.

In [ ]:
EPOCHS           = 50
noise_dim        = 100
num_examples     = 16
NOISE_ANNEAL_END = 30   # epoch at which input noise reaches 0

# Fixed seed so we can visually track the same samples across epochs
seed = tf.random.normal([num_examples, noise_dim])

print(f'Epochs              : {EPOCHS}')
print(f'Batch size          : {BATCH_SIZE}')
print(f'Noise annealing end : epoch {NOISE_ANNEAL_END}')

In [ ]:
# ── IMPROVED train_step ──────────────────────────────────────────────────────
@tf.function
def train_step(images, noise_stddev):
    """
    Improvements over v1:
      1. Input noise added to discriminator inputs (real & fake), annealed over time
      2. Generator is updated TWICE per discriminator update
    """
    batch_size = tf.shape(images)[0]

    # ── Discriminator update (once) ──────────────────────────────────────────
    noise = tf.random.normal([batch_size, noise_dim])

    with tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        # CHANGE 1: add annealing Gaussian noise to both real and fake inputs
        real_noisy = images + tf.random.normal(
            shape=tf.shape(images), stddev=noise_stddev)
        fake_noisy = generated_images + tf.random.normal(
            shape=tf.shape(generated_images), stddev=noise_stddev)

        real_output = discriminator(real_noisy, training=True)
        fake_output = discriminator(fake_noisy, training=True)
        disc_loss   = discriminator_loss(real_output, fake_output)

    disc_grads = disc_tape.gradient(
        disc_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(
        zip(disc_grads, discriminator.trainable_variables))

    # ── Generator update (TWICE) ─────────────────────────────────────────────
    gen_loss = 0.0
    for _ in tf.range(2):   # CHANGE 2: two generator updates per discriminator update
        noise = tf.random.normal([batch_size, noise_dim])
        with tf.GradientTape() as gen_tape:
            generated_images = generator(noise, training=True)
            fake_output      = discriminator(generated_images, training=True)
            gen_loss         = generator_loss(fake_output)

        gen_grads = gen_tape.gradient(
            gen_loss, generator.trainable_variables)
        generator_optimizer.apply_gradients(
            zip(gen_grads, generator.trainable_variables))

    return gen_loss, disc_loss

print('Improved train_step compiled.')

In [ ]:
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(8, 8))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        img = np.clip((predictions[i].numpy() + 1.0) / 2.0, 0, 1)
        plt.imshow(img)
        plt.axis('off')
    plt.suptitle(f'SVHN DCGAN (Improved) — Epoch {epoch:03d}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'image_at_epoch_{epoch:04d}.png', dpi=80)
    plt.show()

In [ ]:
def train(dataset, epochs):
    gen_losses  = []
    disc_losses = []

    for epoch in range(epochs):
        start = time.time()

        # CHANGE: linearly anneal input noise stddev from 0.1 → 0.0 over NOISE_ANNEAL_END epochs
        noise_stddev = max(0.0, 0.1 * (1.0 - epoch / NOISE_ANNEAL_END))

        epoch_gen_loss  = []
        epoch_disc_loss = []

        for image_batch in dataset:
            g_loss, d_loss = train_step(image_batch,
                                        tf.constant(noise_stddev, dtype=tf.float32))
            epoch_gen_loss.append(g_loss.numpy())
            epoch_disc_loss.append(d_loss.numpy())

        display.clear_output(wait=True)
        generate_and_save_images(generator, epoch + 1, seed)

        if (epoch + 1) % 10 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

        avg_gen  = np.mean(epoch_gen_loss)
        avg_disc = np.mean(epoch_disc_loss)
        gen_losses.append(avg_gen)
        disc_losses.append(avg_disc)

        print(f'Epoch {epoch+1:3d}/{epochs} | '
              f'Gen Loss: {avg_gen:.4f} | '
              f'Disc Loss: {avg_disc:.4f} | '
              f'Input noise σ: {noise_stddev:.3f} | '
              f'Time: {time.time()-start:.1f}s')

    display.clear_output(wait=True)
    generate_and_save_images(generator, epochs, seed)
    return gen_losses, disc_losses

## 8. Run Training

> **Recommended:** Run on Google Colab with GPU (Runtime → Change runtime type → T4 GPU). Expect ~1–2 min/epoch.

In [ ]:
%%time
gen_losses, disc_losses = train(train_dataset, EPOCHS)

## 9. Loss Curves Analysis

With the improvements, the expected behavior compared to the original:
- **Discriminator loss** should start higher (~1.0–1.3 instead of ~0.4) because label smoothing and input noise prevent immediate overconfidence
- **Generator loss** should begin declining earlier (before epoch 10 instead of plateauing until epoch 40)
- Both losses should converge closer together, ideally within ~0.5 of each other

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: improved losses
axes[0].plot(range(1, EPOCHS+1), gen_losses,  label='Generator',     color='blue',   lw=2)
axes[0].plot(range(1, EPOCHS+1), disc_losses, label='Discriminator', color='orange', lw=2)
axes[0].set_title('Improved DCGAN — Training Losses')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: gap between losses (healthy = small)
gap = np.array(gen_losses) - np.array(disc_losses)
axes[1].plot(range(1, EPOCHS+1), gap, color='purple', lw=2)
axes[1].axhline(0, color='black', linestyle='--', alpha=0.4)
axes[1].fill_between(range(1, EPOCHS+1), gap, 0,
                     where=gap > 0, alpha=0.15, color='blue',   label='Gen leading')
axes[1].fill_between(range(1, EPOCHS+1), gap, 0,
                     where=gap < 0, alpha=0.15, color='orange', label='Disc leading')
axes[1].set_title('Loss Gap (Gen - Disc)  →  smaller = more balanced')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss difference')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('improved_training_losses.png', dpi=100)
plt.show()

print(f'Final Generator Loss     : {gen_losses[-1]:.4f}')
print(f'Final Discriminator Loss : {disc_losses[-1]:.4f}')
print(f'Final Gap                : {gen_losses[-1] - disc_losses[-1]:.4f}')

## 10. Restore Latest Checkpoint & View Results

In [ ]:
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))
print('Latest checkpoint restored.')

In [ ]:
PIL.Image.open(f'image_at_epoch_{EPOCHS:04d}.png')

## 11. Animated GIF of Training Progression

In [ ]:
anim_file = 'dcgan_svhn_improved.gif'

with imageio.get_writer(anim_file, mode='I', duration=0.2) as writer:
    filenames = sorted(glob.glob('image_at_epoch_*.png'))
    for filename in filenames:
        writer.append_data(imageio.v2.imread(filename))
    writer.append_data(imageio.v2.imread(filenames[-1]))  # pause on last frame

print(f'GIF saved: {anim_file}')

import IPython
IPython.display.Image(filename=anim_file)

## 12. Final Generated Image Grid

In [ ]:
new_noise  = tf.random.normal([64, noise_dim])
new_images = generator(new_noise, training=False)

fig, axes = plt.subplots(8, 8, figsize=(16, 16))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(np.clip((new_images[i].numpy() + 1.0) / 2.0, 0, 1))
    ax.axis('off')

plt.suptitle(f'64 Generated Images — Improved DCGAN (Epoch {EPOCHS})', fontsize=14)
plt.tight_layout()
plt.savefig('final_generated_improved.png', dpi=100)
plt.show()

## 13. Summary of All Changes (v1 → v2)

| Component | v1 (Original) | v2 (Improved) | Why |
|---|---|---|---|
| **Real labels** | `1.0` | `0.9` (label smoothing) | Prevents discriminator overconfidence |
| **Optimizer beta_1** | `0.9` (default) | `0.5` | Reduces momentum overshoot in adversarial setting |
| **Input noise** | None | Gaussian σ=0.1→0.0 over 30 epochs | Slows early discriminator dominance |
| **Generator updates** | 1× per step | 2× per step | Rebalances adversarial dynamic |
| **Generator depth** | 3 Conv2DTranspose | 4 Conv2DTranspose (64-filter layer added) | More capacity for fine color detail |
| **Loss monitoring** | Single plot | Loss + gap plot | Better diagnostic visibility |

### What to look for in the results
- Discriminator loss starting around **1.0–1.3** (vs ~0.4 in v1) confirms label smoothing is working
- Generator loss declining **before epoch 10** (vs plateau until epoch 40 in v1) confirms the 2× update rule is effective
- The **loss gap chart** should show a narrowing gap toward equilibrium over the 50 epochs